# 04 — Donchian Breakout (Turtle)

Classic 20/10 channel breakout on a small ETF panel.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.backtest.runner import run
from src.data.store import get_or_fetch_panel


In [ ]:
from src.strategies.donchian import Donchian

close = get_or_fetch_panel(['SPY', 'QQQ', 'GLD', 'DBC'], '2010-01-01')
close.tail(3)


In [ ]:
strat = Donchian(entry_lookback=20, exit_lookback=10)
result = run(close, strat)
print(result.summary())


In [ ]:
pf = result.portfolio
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pf.value().plot(ax=axes[0]); axes[0].set_title('Equity')
pf.drawdown().plot(ax=axes[1], color='red'); axes[1].set_title('Drawdown')
plt.tight_layout(); plt.show()


### Channel + breakout markers (SPY)


In [ ]:
spy = close['SPY']
high20 = spy.shift(1).rolling(20).max()
low10 = spy.shift(1).rolling(10).min()
sig = strat.signals(close)['SPY']
fig, ax = plt.subplots(figsize=(11, 5))
tail = slice(-500, None)
spy.iloc[tail].plot(ax=ax, label='SPY')
high20.iloc[tail].plot(ax=ax, color='green', alpha=0.4, label='20d high')
low10.iloc[tail].plot(ax=ax, color='red', alpha=0.4, label='10d low')
ax.fill_between(spy.iloc[tail].index, spy.iloc[tail].min(), spy.iloc[tail].max(),
                where=sig.iloc[tail], color='green', alpha=0.08)
ax.legend(); ax.set_title('Donchian channel — SPY'); plt.show()
